# E3-F2A — Prolongation contrôlée des GCN (200 époques)

**Objectif :** vérifier si les résultats des GCN de E3-D sont limités par le budget initial de 80 époques, sans altérer les résultats historiques.

- 4 horizons × 3 topologies × 3 graines = **36 entraînements indépendants**.
- **Nouvel entraînement depuis l'initialisation** (et non reprise exacte du checkpoint) : les anciens checkpoints ne sauvegardent pas l'état Adam.
- Même architecture, mêmes hyperparamètres, mêmes graines, mêmes données et topologies que E3-D ; largeur cachée **figée selon la sélection E3-D** (aucune nouvelle recherche).
- `MAX_EPOCHS=200`, `PATIENCE=25` ; sélection du meilleur checkpoint **sur validation uniquement**.
- Prédictions test produites après entraînement ; 13 643 clés communes.
- Sorties distinctes dans `outputs/E3_F2A_GCN_200epochs/`.

**Limite méthodologique :** ce protocole est un *réentraînement à budget prolongé*, non la continuation exacte d'un état optimiseur. Les différences avec E3-D doivent être interprétées en conséquence.

In [ ]:
from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd
import networkx as nx
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
OLD=ROOT/'outputs/E3_10D_GCN_multihorizon'
OUT=ROOT/'outputs/E3_F2A_GCN_200epochs'
HORIZONS=[15,30,60,120]
SPLITS=['train','validation','test']
SEEDS=[42,123,2026]
NODES,SEQ,CHANNELS=7,5,3
BATCH_SIZE=256
MAX_EPOCHS=200
PATIENCE=25
LR=1e-3
WEIGHT_DECAY=1e-5
K_NEIGHBORS=2
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert REP.is_dir() and COMMON.is_dir(), 'Missing E3 aligned representations/common indices.'
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common['timestamp'],utc=True)
assert len(common)==91763
assert common.groupby('split').size().to_dict()=={'train':64393,'validation':13727,'test':13643}
assert not common.duplicated(['uuid','timestamp','split']).any()
print('Root:',ROOT,'Device:',device,'Torch:',torch.__version__)

assert OLD.is_dir(), f'Original E3-D folder missing: {OLD}'
old_configs=pd.read_csv(OLD/'05_selected_configs_and_training.csv')
old_protocol=json.loads((OLD/'06_protocol.json').read_text(encoding='utf-8'))
assert old_protocol['max_epochs']==80
assert set(old_configs['topology'])=={'G_native','G_operational','G_shuffled'}
assert len(old_configs)==36
SELECTED={}
for h in HORIZONS:
    rows=old_configs.loc[old_configs.horizon_min.eq(h)]
    widths=rows.hidden.unique().tolist()
    assert len(rows)==9 and len(widths)==1, f'Inconsistent frozen width h={h}: {widths}'
    SELECTED[h]=int(widths[0])
print('Frozen hidden widths:',SELECTED)


In [ ]:
def load_horizon(h):
    folder=REP/f'h{h:03d}'
    nodes=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    assert nodes['node_index'].tolist()==list(range(7))
    devices=nodes['uuid'].astype(str).tolist()
    data={}
    for split in SPLITS:
        X=np.load(folder/f'X_graph_{split}_scaled.npy').astype(np.float32)
        Xu=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        y=np.load(folder/f'y_graph_{split}.npy').astype(np.float32)
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        meta['timestamp']=pd.to_datetime(meta['timestamp'],utc=True)
        meta['target_timestamp']=pd.to_datetime(meta['target_timestamp'],utc=True)
        idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy').astype(np.int64)
        assert X.shape==(len(y),5,7,3) and y.shape[1]==7
        assert len(meta)==y.size==len(idx)*0+7*len(y)
        assert len(idx)=={'train':64393,'validation':13727,'test':13643}[split]
        assert len(set(idx.tolist()))==len(idx) and idx.min()>=0 and idx.max()<y.size
        assert meta['uuid'].astype(str).tolist()==devices*len(y), 'Unexpected node order.'
        assert meta['split'].eq(split).all()
        assert (meta['target_timestamp']-meta['timestamp']).eq(pd.Timedelta(minutes=h)).all()
        assert np.allclose(meta['target'].to_numpy(),y.reshape(-1),rtol=0,atol=1e-5)
        assert np.isfinite(X).all() and np.isfinite(y).all()
        assert np.allclose(Xu.transpose(0,2,1,3).reshape(-1,5,3)[idx],
                           np.load(folder/f'X_sequence_{split}_unscaled.npy',mmap_mode='r')[idx],
                           rtol=0,atol=0)
        selected=meta.iloc[idx][['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        expected=common.loc[common.split.eq(split),['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        assert selected.equals(expected),f'Common keys differ h={h} split={split}'
        data[split]=dict(X=X,y=y,meta=meta,idx=idx,
                         persistence=np.asarray(Xu[:,-1,:,0],dtype=np.float64).reshape(-1))
        print(h,split,X.shape,'common nodes',len(idx))
    return data,devices

def build_topologies(train_X, h):
    # Multivariate train-only similarity: flatten all time snapshots and history
    # for each variable; standardized features are already train-only scaled.
    # Pearson correlation computed separately per variable, then averaged.
    v=train_X.transpose(2,0,1,3).reshape(7,-1,3)
    similarity=np.zeros((7,7),dtype=np.float64)
    for feature in range(3):
        z=v[:,:,feature]
        corr=np.corrcoef(z)
        assert np.isfinite(corr).all(), 'Constant training trajectory; inspect topology.'
        similarity+=corr/3
    np.fill_diagonal(similarity,-np.inf)
    edges=set()
    for i in range(7):
        order=sorted((j for j in range(7) if j!=i),
                     key=lambda j:(-similarity[i,j],j))
        for j in order[:K_NEIGHBORS]:
            edges.add(tuple(sorted((i,j))))
    g=nx.Graph();g.add_nodes_from(range(7));g.add_edges_from(sorted(edges))
    # Degree-preserving null topology. Fail if no genuine rewiring possible.
    shuffled=None
    for trial in range(100):
        cand=g.copy()
        try:
            nx.double_edge_swap(cand,nswap=2,max_tries=1000,seed=42+h+trial)
        except nx.NetworkXAlgorithmError:
            continue
        if set(cand.edges())!=set(g.edges()) and sorted(dict(cand.degree()).items())==sorted(dict(g.degree()).items()):
            shuffled=cand
            break
    assert shuffled is not None,'Could not construct a distinct degree-preserving shuffled topology.'
    assert dict(g.degree())==dict(shuffled.degree())
    def Ahat(graph):
        A=nx.to_numpy_array(graph,nodelist=list(range(7)),dtype=np.float32)
        A+=np.eye(7,dtype=np.float32)
        d=A.sum(axis=1)
        return torch.tensor(A/np.sqrt(d[:,None]*d[None,:]),dtype=torch.float32,device=device)
    topologies={'G_native':nx.empty_graph(7),
                'G_operational':g,
                'G_shuffled':shuffled}
    rows=[]
    for name,graph in topologies.items():
        for i,j in graph.edges():
            rows.append({'horizon_min':h,'topology':name,'node_i':i,'node_j':j})
    pd.DataFrame(rows,columns=['horizon_min','topology','node_i','node_j']).to_csv(
        OUT/f'h{h:03d}_topology_edges.csv',index=False)
    return {k:Ahat(v) for k,v in topologies.items()}, {
        k:sorted(tuple(sorted(e)) for e in v.edges()) for k,v in topologies.items()
    }


In [ ]:
class GraphConv(nn.Module):
    def __init__(self,fin,fout):
        super().__init__()
        self.lin=nn.Linear(fin,fout)
    def forward(self,x,Ahat):
        return self.lin(torch.einsum('ij,bjf->bif',Ahat,x))

class TemporalGCN(nn.Module):
    def __init__(self,hidden=32,dropout=0.1):
        super().__init__()
        self.gc1=GraphConv(15,hidden)
        self.gc2=GraphConv(hidden,hidden)
        self.drop=nn.Dropout(dropout)
        self.head=nn.Linear(hidden,1)
    def forward(self,x,Ahat):
        z=x.permute(0,2,1,3).contiguous().reshape(x.size(0),7,15)
        z=self.drop(torch.relu(self.gc1(z,Ahat)))
        z=self.drop(torch.relu(self.gc2(z,Ahat)))
        return self.head(z).squeeze(-1)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass

def loader(data,split,shuffle=False):
    d=data[split]
    return DataLoader(TensorDataset(torch.from_numpy(d['X']),torch.from_numpy(d['y'])),
                      batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0)

@torch.no_grad()
def predict(model,data,split,Ahat):
    model.eval()
    return np.concatenate([model(x.to(device),Ahat).cpu().numpy()
                           for x,_ in loader(data,split)],axis=0)

def fit(hidden,Ahat,data,seed,ckpt):
    seed_all(seed)  # Seed before model initialization
    model=TemporalGCN(hidden,0.1).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    best=float('inf');bad=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();losses=[]
        for x,y in loader(data,'train',shuffle=True):
            x,y=x.to(device),y.to(device)
            opt.zero_grad()
            loss=nn.functional.mse_loss(model(x,Ahat),y)
            loss.backward();opt.step()
            losses.append(loss.item())
        p=predict(model,data,'validation',Ahat)
        rmse=float(np.sqrt(np.mean((p-data['validation']['y'])**2)))
        history.append({'epoch':epoch,'train_MSE':float(np.mean(losses)),
                        'validation_RMSE':rmse})
        if rmse<best-1e-7:
            best=rmse;bad=0
            torch.save(model.state_dict(),ckpt)
        else:
            bad+=1
            if bad>=PATIENCE:break
    model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
    return model,pd.DataFrame(history),best

def metrics(y,p):
    y=np.asarray(y,dtype=np.float64);p=np.asarray(p,dtype=np.float64)
    e=p-y;den=np.sum((y-y.mean())**2)
    return {'n':len(y),'RMSE':float(np.sqrt(np.mean(e*e))),
            'MAE':float(np.mean(np.abs(e))),
            'R2':float(1-np.sum(e*e)/den) if den>0 else np.nan,
            'Bias':float(e.mean())}


## Contrôle préalable des topologies

Le graphe opérationnel est reconstruit exclusivement à partir de l'entraînement. Les arêtes obtenues sont comparées à celles du manifeste original E3-D. En cas de divergence, le notebook s'arrête : la comparaison ne serait plus une simple prolongation du budget.

In [ ]:
run_rows=[];pred_frames=[];device_rows=[];config_rows=[]
topology_manifest={}
for h in HORIZONS:
    print(f'\n===== HORIZON {h} min =====')
    data,devices=load_horizon(h)
    Ahats,edges=build_topologies(data['train']['X'],h)
    topology_manifest[str(h)]=edges
    for topology in ('G_native','G_operational','G_shuffled'):
        original=[tuple(sorted(e)) for e in old_protocol['topologies'][str(h)][topology]]
        assert sorted(edges[topology])==sorted(original),f'Topology changed: h={h}, {topology}'
    selected=SELECTED[h]
    print('Frozen hidden width:',selected)
    for topology in ['G_native','G_operational','G_shuffled']:
        Ahat=Ahats[topology]
        for seed in SEEDS:
            start=time.perf_counter()
            ckpt=OUT/f'h{h:03d}_{topology}_seed{seed}.pt'
            model,hist,best=fit(selected,Ahat,data,seed,ckpt)
            hist.to_csv(OUT/f'h{h:03d}_{topology}_seed{seed}_history.csv',index=False)
            config_rows.append({'horizon_min':h,'topology':topology,'hidden':selected,
                                'seed':seed,'best_validation_RMSE':best,
                                'best_epoch':int(hist.loc[hist.validation_RMSE.idxmin(),'epoch']),
                                'epochs_run':len(hist),
                                'training_seconds':time.perf_counter()-start,
                                'train_snapshots':len(data['train']['y']),
                                'validation_snapshots':len(data['validation']['y']),
                                'test_snapshots':len(data['test']['y'])})
            for split in SPLITS:
                d=data[split]
                p=predict(model,data,split,Ahat).reshape(-1)
                y=d['y'].reshape(-1)
                assert len(p)==len(y)==len(d['meta']) and np.isfinite(p).all()
                for scope,idx in [('full_horizon_split',np.arange(len(y))),
                                  ('common_keys',d['idx'])]:
                    if scope=='common_keys' and split!='test':continue
                    m=metrics(y[idx],p[idx]);r=metrics(y[idx],d['persistence'][idx])
                    run_rows.append({'horizon_min':h,'topology':topology,'seed':seed,
                                     'split':split,'scope':scope,**m,
                                     'Persistence_RMSE':r['RMSE'],
                                     'skill_RMSE':1-m['RMSE']/r['RMSE']})
                if split=='test':
                    idx=d['idx'];df=d['meta'].iloc[idx].copy().reset_index(drop=True)
                    df['horizon_min']=h;df['topology']=topology;df['seed']=seed
                    df['y_true']=y[idx];df['y_pred']=p[idx]
                    df['persistence_pred']=d['persistence'][idx];df['error']=p[idx]-y[idx]
                    pred_frames.append(df)
                    for uuid,g in df.groupby('uuid'):
                        m=metrics(g.y_true,g.y_pred);r=metrics(g.y_true,g.persistence_pred)
                        device_rows.append({'horizon_min':h,'topology':topology,
                                            'seed':seed,'uuid':uuid,**m,
                                            'Persistence_RMSE':r['RMSE'],
                                            'skill_RMSE':1-m['RMSE']/r['RMSE']})
            print('Completed',h,topology,seed,'best validation',round(best,5),
                  'best epoch',config_rows[-1]['best_epoch'])
            del model
            if torch.cuda.is_available():torch.cuda.empty_cache()

runs=pd.DataFrame(run_rows);preds=pd.concat(pred_frames,ignore_index=True)
per_device=pd.DataFrame(device_rows);configs=pd.DataFrame(config_rows)
primary=runs.query("split=='test' and scope=='common_keys'").copy()
assert len(configs)==36 and len(primary)==36
assert len(preds)==4*3*3*13643
assert not preds.duplicated(['horizon_min','topology','seed','uuid','timestamp']).any()
assert primary['n'].eq(13643).all()
summary=(primary.groupby(['horizon_min','topology'],as_index=False)
         .agg(n_seeds=('seed','nunique'),n=('n','first'),
              RMSE_mean=('RMSE','mean'),RMSE_SD=('RMSE','std'),
              MAE_mean=('MAE','mean'),MAE_SD=('MAE','std'),
              R2_mean=('R2','mean'),R2_SD=('R2','std'),
              Bias_mean=('Bias','mean'),Bias_SD=('Bias','std'),
              skill_RMSE_mean=('skill_RMSE','mean'),
              skill_RMSE_SD=('skill_RMSE','std')))
assert summary.n_seeds.eq(3).all()
primary.to_csv(OUT/'00_PRIMARY_common_TEST_per_seed.csv',index=False)
summary.to_csv(OUT/'00_PRIMARY_common_TEST_mean_sd.csv',index=False)
runs.to_csv(OUT/'02_metrics_all_seeds_all_splits.csv',index=False)
preds.to_csv(OUT/'03_common_test_predictions_all_seeds.csv',index=False)
per_device.to_csv(OUT/'04_common_test_per_device_per_seed.csv',index=False)
configs.to_csv(OUT/'05_selected_configs_and_training.csv',index=False)
protocol={'experiment':'E3-F2A controlled extended-budget GCN rerun',
          'source':'E3_aligned_representations','horizons_minutes':HORIZONS,
          'seeds':SEEDS,'selected_hidden_frozen_from_E3_D':SELECTED,
          'topologies':topology_manifest,
          'original_protocol_file':str(OLD/'06_protocol.json'),
          'training_mode':'new run from same seed and initialization, not optimizer-state continuation',
          'test':'13643 common node samples per horizon/topology/seed',
          'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
          'batch_size':BATCH_SIZE,'optimizer':'Adam',
          'learning_rate':LR,'weight_decay':WEIGHT_DECAY,
          'historical_results':'unchanged',
          'python':platform.python_version(),'torch':torch.__version__,
          'networkx':nx.__version__}
(OUT/'06_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(summary.to_string(index=False));print('Saved:',OUT)


## À transmettre

Compresser `outputs/E3_F2A_GCN_200epochs/`, ou transmettre les fichiers `00_PRIMARY_common_TEST_mean_sd.csv`, `00_PRIMARY_common_TEST_per_seed.csv`, `05_selected_configs_and_training.csv`, `06_protocol.json` et les 36 historiques `*_history.csv`.

Les prédictions détaillées `03_common_test_predictions_all_seeds.csv` seront nécessaires pour les tests statistiques appariés ultérieurs.

**Point de vigilance :** les anciens résultats E3-D doivent être conservés ; si les performances changent, l'attribution est au *budget d'optimisation*, et non automatiquement au paradigme graphique.